# AI-Powered Medical Image Analyzer: MobileNetV2 Transfer Learning

This notebook trains a binary classification model to detect **PNEUMONIA** vs. **NORMAL** from chest X-ray images using **MobileNetV2** transfer learning.

**Workflow:**
1. Load and explore the dataset (`train`, `val`, `test`)
2. Data augmentation & MobileNetV2 preprocessing (224x224, [-1, 1] scaling)
3. Base model setup (MobileNetV2 pretrained on ImageNet)
4. Feature extraction training (frozen base)
5. Fine-tuning top layers with low learning rate
6. Evaluation on test set (Confusion Matrix, Precision, Recall, F1)
7. Export the trained model to `../backend/models/medical_model.keras`

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

print('TensorFlow Version:', tf.__version__)
print('GPU Available:', tf.config.list_physical_devices('GPU'))

## 1. Dataset Path Discovery and Configuration
We dynamically detect the dataset location relative to the project or workspace.

In [ ]:
possible_dirs = [
    Path('../dataset/chest_xray'),
    Path('../../dataset/chest_xray'),
    Path('dataset/chest_xray')
]
DATASET_DIR = None
for p in possible_dirs:
    if p.exists() and (p / 'train').exists():
        DATASET_DIR = p.resolve()
        break

if DATASET_DIR is None:
    raise FileNotFoundError('Chest X-Ray dataset directory not found. Please verify folder structure.')

TRAIN_DIR = DATASET_DIR / 'train'
VAL_DIR = DATASET_DIR / 'val'
TEST_DIR = DATASET_DIR / 'test'

print('Using dataset directory:', DATASET_DIR)
print('Train counts - NORMAL:', len(list((TRAIN_DIR / 'NORMAL').glob('*.jpeg'))), 'PNEUMONIA:', len(list((TRAIN_DIR / 'PNEUMONIA').glob('*.jpeg'))))
print('Val counts   - NORMAL:', len(list((VAL_DIR / 'NORMAL').glob('*.jpeg'))), 'PNEUMONIA:', len(list((VAL_DIR / 'PNEUMONIA').glob('*.jpeg'))))
print('Test counts  - NORMAL:', len(list((TEST_DIR / 'NORMAL').glob('*.jpeg'))), 'PNEUMONIA:', len(list((TEST_DIR / 'PNEUMONIA').glob('*.jpeg'))))

## 2. Data Loading & Pipeline Configuration
MobileNetV2 requires 224x224 RGB inputs. Because the default `val` folder in this dataset only contains 16 images, we also incorporate validation splitting or use `val` / `test` appropriately for robust validation.

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
    label_mode='binary'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    label_mode='binary'
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    label_mode='binary'
)

class_names = train_ds.class_names
print('Class labels:', class_names)

### Handle Class Imbalance
Notice that the dataset has significantly more PNEUMONIA cases (~3875) than NORMAL (~1341). We compute class weights to balance gradient updates during training.

In [ ]:
count_normal = len(list((TRAIN_DIR / 'NORMAL').glob('*.jpeg')))
count_pneumonia = len(list((TRAIN_DIR / 'PNEUMONIA').glob('*.jpeg')))
total_samples = count_normal + count_pneumonia

weight_for_0 = (1 / count_normal) * (total_samples / 2.0)
weight_for_1 = (1 / count_pneumonia) * (total_samples / 2.0)
class_weight = {0: weight_for_0, 1: weight_for_1}
print(f'Class Weight -> 0 ({class_names[0]}): {weight_for_0:.2f}, 1 ({class_names[1]}): {weight_for_1:.2f}')

### Data Augmentation & Preprocessing
We apply standard data augmentation to improve generalization and MobileNetV2 preprocessing (scales pixel values to [-1, 1]).

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal', seed=SEED),
    layers.RandomRotation(0.08, seed=SEED),
    layers.RandomZoom(0.08, seed=SEED),
], name='data_augmentation')

# Optimize data pipeline caching and prefetching
AUTOTUNE = tf.data.AUTOTUNE
train_ds_proc = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds_proc = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds_proc = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

## 3. Build Model Architecture (MobileNetV2 Base + Classification Head)
- Base: MobileNetV2 pretrained on ImageNet (`weights='imagenet'`, `include_top=False`)
- Base layers initially frozen (`base_model.trainable = False`)
- Classification Head: GlobalAveragePooling2D -> Dropout(0.3) -> Dense(1, activation='sigmoid')

In [ ]:
inputs = keras.Input(shape=(224, 224, 3), name='input_layer')
x = data_augmentation(inputs)
x = preprocess_input(x)

base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D(name='avg_pool')(x)
x = layers.Dropout(0.3, name='top_dropout')(x)
outputs = layers.Dense(1, activation='sigmoid', name='predictions')(x)

model = keras.Model(inputs, outputs, name='Chest_XRay_MobileNetV2')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=['accuracy', keras.metrics.Precision(name='precision'), keras.metrics.Recall(name='recall')]
)

model.summary()

## 4. Phase 1: Train Classification Head (Feature Extraction)
Train the newly added top layer while keeping the base feature extractor frozen.

In [ ]:
# Setup output paths
possible_model_dirs = [Path('../backend/models'), Path('backend/models')]
MODEL_DIR = next((p.resolve() for p in possible_model_dirs if p.parent.exists()), Path('../backend/models').resolve())
MODEL_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_PATH = MODEL_DIR / 'medical_model.keras'

callbacks_phase1 = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1)
]

EPOCHS_HEAD = 5
print(f'Starting Phase 1 training for {EPOCHS_HEAD} epochs...')
history_phase1 = model.fit(
    train_ds_proc,
    validation_data=test_ds_proc,
    epochs=EPOCHS_HEAD,
    class_weight=class_weight,
    callbacks=callbacks_phase1
)

## 5. Phase 2: Fine-Tuning Top Layers
Unfreeze only the top convolutional layers of MobileNetV2 (from layer 120 onwards) and train with a significantly smaller learning rate (`1e-5`) so as not to destroy pretrained weights.

In [ ]:
base_model.trainable = True
# Freeze all layers before index 120 (fine-tune the top layers only)
fine_tune_at = 120
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f'Total layers in base model: {len(base_model.layers)}')
print(f'Trainable layers in base model: {trainable_count}')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=['accuracy', keras.metrics.Precision(name='precision'), keras.metrics.Recall(name='recall')]
)

callbacks_phase2 = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
    ModelCheckpoint(filepath=str(BEST_MODEL_PATH), monitor='val_loss', save_best_only=True, verbose=1)
]

EPOCHS_FINE = 5
print(f'Starting Phase 2 Fine-Tuning for {EPOCHS_FINE} epochs...')
history_phase2 = model.fit(
    train_ds_proc,
    validation_data=test_ds_proc,
    epochs=EPOCHS_FINE,
    class_weight=class_weight,
    callbacks=callbacks_phase2
)

## 6. Training Visualization
Plot training and validation accuracy and loss across training phases.

In [ ]:
def plot_metrics(h1, h2):
    acc = h1.history['accuracy'] + h2.history['accuracy']
    val_acc = h1.history['val_accuracy'] + h2.history['val_accuracy']
    loss = h1.history['loss'] + h2.history['loss']
    val_loss = h1.history['val_loss'] + h2.history['val_loss']

    plt.figure(figsize=(14, 5))
    plt.subplot(1, 2, 1)
    plt.plot(acc, label='Training Accuracy', marker='o')
    plt.plot(val_acc, label='Validation Accuracy', marker='o')
    plt.axvline(x=len(h1.history['accuracy']) - 1, color='red', linestyle='--', label='Fine Tuning Start')
    plt.title('Accuracy Over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(loss, label='Training Loss', marker='o')
    plt.plot(val_loss, label='Validation Loss', marker='o')
    plt.axvline(x=len(h1.history['loss']) - 1, color='red', linestyle='--', label='Fine Tuning Start')
    plt.title('Loss Over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_metrics(history_phase1, history_phase2)

## 7. Model Evaluation on Test Set
We evaluate on the unseen test set and compute:
- Test Accuracy & Loss
- Precision, Recall, and F1-Score
- Confusion Matrix display

In [ ]:
y_true = []
y_pred_probs = []

for images, labels in test_ds:
    probs = model.predict(images, verbose=0)
    y_true.extend(labels.numpy().flatten())
    y_pred_probs.extend(probs.flatten())

y_true = np.array(y_true, dtype=int)
y_pred_probs = np.array(y_pred_probs)
y_pred = (y_pred_probs >= 0.5).astype(int)

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(cmap=plt.cm.Blues, ax=ax)
plt.title('Test Set Confusion Matrix')
plt.show()

## 8. Export Model & Inference Configuration
Save the final trained weights and inference metadata for FastAPI.

In [ ]:
# Save final model if not already saved by checkpoint
model.save(str(BEST_MODEL_PATH))
print(f'Saved final model to: {BEST_MODEL_PATH}')

# Save inference configuration
config_path = MODEL_DIR / 'model_config.json'
config = {
    'target_size': [224, 224],
    'class_names': list(class_names),
    'architecture': 'MobileNetV2',
    'threshold': 0.5,
    'last_conv_layer_name': 'Conv_1'
}

with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f'Saved configuration to: {config_path}')